In [7]:
import os
import re
import time
import pandas as pd
from Bio import Entrez, SeqIO
from Bio.SeqUtils.ProtParam import ProteinAnalysis

# Step A: Identify Sample_id and tool

In [8]:

FILENAME_PATTERN = re.compile(
    r"^(?P<sample_id>.+?)_(?P<tool>fusioncatcher|arriba)_case_specific_filtered\.tsv$"
)


def parse_filename(filename):

    match = FILENAME_PATTERN.match(filename)

    if match is None:
        return None, None

    return match.group("sample_id"), match.group("tool")


# Step B: ARRIBA fusion_transcript Cleaner

In [9]:
def clean_arriba_fusion_transcript(fusion_transcript):

    if fusion_transcript is None or fusion_transcript.strip() in ("", "."):
        return None, None, None, False

    sequence = fusion_transcript.strip()

    truncated = False

    if "..." in sequence:
        sequence = sequence.split("...")[0]
        truncated = True

    parts = sequence.split("|")

    if len(parts) == 2:
        left_raw, right_raw = parts
        insertion_raw = ""

    elif len(parts) == 3:
        left_raw, insertion_raw, right_raw = parts

    else:
        
        return None, None, None, truncated

    def clean_segment(segment):
        segment = segment.replace("___", "")     
        segment = segment.replace("-", "")       
        segment = segment.replace("[", "").replace("]", "") 
        segment = segment.replace("?", "N")    
        segment = segment.upper()
        return segment

    left_fragment = clean_segment(left_raw)
    right_fragment = clean_segment(right_raw)
    insertion_sequence = clean_segment(insertion_raw)

    return left_fragment, right_fragment, insertion_sequence, truncated


# Step C: Raw adapters per tool

In [10]:
def parse_fusioncatcher_row(row, sample_id):

    dna_fs = row.get("Fusion_sequence", "")

    return {

        "Sample_ID": sample_id,
        "Tool": "fusioncatcher",
        "Gene_A_name": row.get("Gene_1_symbol(5end_fusion_partner)", ""),
        "Gene_B_name": row.get("Gene_2_symbol(3end_fusion_partner)", ""),
        "Gene_A_ensembl_id": row.get("Gene_1_id(5end_fusion_partner)", ""),
        "Gene_B_ensembl_id": row.get("Gene_2_id(3end_fusion_partner)", ""),
        "dna_fs": dna_fs,
        "Insertion_sequence": "",
        "reported_frame": row.get("Predicted_effect", ""),
        "Spanning_unique_reads": row.get("Spanning_unique_reads", ""),
        "Spanning_pairs": row.get("Spanning_pairs", ""),
        "Truncated_due_to_low_coverage": False,
        "Parse_successful": dna_fs not in ("", None) and "*" in dna_fs,

    }


def parse_arriba_row(row, sample_id):

    fusion_transcript = row.get("fusion_transcript", "")

    left_fragment, right_fragment, insertion_sequence, truncated = \
        clean_arriba_fusion_transcript(fusion_transcript)

    parse_successful = left_fragment is not None and right_fragment is not None

    if parse_successful:
        dna_fs = left_fragment + "*" + right_fragment
    else:
        dna_fs = ""

    return {

        "Sample_ID": sample_id,
        "Tool": "arriba",
        "Gene_A_name": row.get("gene1", ""),
        "Gene_B_name": row.get("gene2", ""),
        "Gene_A_ensembl_id": row.get("gene_id1", ""),
        "Gene_B_ensembl_id": row.get("gene_id2", ""),
        "dna_fs": dna_fs,
        "Insertion_sequence": insertion_sequence if insertion_sequence else "",
        "reported_frame": row.get("reading_frame", ""),
        "Confidence": row.get("confidence", ""),
        "Arriba_peptide_sequence": row.get("peptide_sequence", ""),
        "Split_reads1": row.get("split_reads1", ""),
        "Split_reads2": row.get("split_reads2", ""),
        "Truncated_due_to_low_coverage": truncated,
        "Parse_successful": parse_successful,

    }



# Step D: Scan the sample files folder and build a master table

In [11]:
def load_all_sample_fusions(folder_path):

    all_fusion_records = []

    for filename in sorted(os.listdir(folder_path)):

        sample_id, tool = parse_filename(filename)

        if sample_id is None:
            continue   # not a recognized fusion file, skip it

        file_path = os.path.join(folder_path, filename)

        df = pd.read_csv(file_path, sep="\t")

      
        df.columns = [column.lstrip("#") for column in df.columns]

        for _, row in df.iterrows():

            if tool == "fusioncatcher":
                record = parse_fusioncatcher_row(row, sample_id)
            else:
                record = parse_arriba_row(row, sample_id)

            all_fusion_records.append(record)

    master_fusion_table = pd.DataFrame(all_fusion_records)

    return master_fusion_table


# Run

if __name__ == "__main__":

    folder_path = input("Enter path to the folder containing per-sample fusion TSVs: ")

    master_fusion_table = load_all_sample_fusions(folder_path)

    print("\nTotal fusion records loaded:", len(master_fusion_table))
    print("\nBy tool:")
    print(master_fusion_table["Tool"].value_counts())
    print("\nBy sample:")
    print(master_fusion_table["Sample_ID"].value_counts())

    print("\nParse failures:")
    print(master_fusion_table[~master_fusion_table["Parse_successful"]][["Sample_ID", "Tool"]].value_counts())

    master_fusion_table.to_csv("master_fusion_table.csv", index=False)
    print("\nSaved as master_fusion_table.csv")

Enter path to the folder containing per-sample fusion TSVs:  /home/jerrybryt/MSBT/IP/Gene_Fusions/Case_specific_fusions/Case_specific_fusions



Total fusion records loaded: 1975

By tool:
Tool
fusioncatcher    1408
arriba            567
Name: count, dtype: int64

By sample:
Sample_ID
AKU076     179
KAIC016    176
KAIC029    142
AKU066     141
AKU082     131
KAIC023    130
KAIC027    105
KAIC010     93
KAIC018     84
KAIC009     83
KAIC026     78
AKU040      77
AKU085      68
KAIC030     66
AKU073      63
AKU089      63
KAIC012     58
AKU041      55
AKU087      48
KAIC017     46
KAIC014     33
KAIC032     28
KAIC033     28
Name: count, dtype: int64

Parse failures:
Sample_ID  Tool  
KAIC029    arriba    15
KAIC010    arriba    11
KAIC012    arriba    11
KAIC023    arriba    11
AKU082     arriba     9
AKU076     arriba     8
KAIC016    arriba     8
AKU066     arriba     7
KAIC027    arriba     6
AKU041     arriba     4
KAIC009    arriba     4
AKU073     arriba     3
AKU089     arriba     3
KAIC018    arriba     3
KAIC026    arriba     3
AKU085     arriba     2
KAIC014    arriba     2
KAIC030    arriba     2
KAIC032    arriba   

In [17]:

Entrez.email = "ojeromebright@gmail.com"


genetic_code = {

    "ATA":"I","ATC":"I","ATT":"I","ATG":"M",
    "ACA":"T","ACC":"T","ACG":"T","ACT":"T",
    "AAC":"N","AAT":"N","AAA":"K","AAG":"K",
    "AGC":"S","AGT":"S","AGA":"R","AGG":"R",
    "CTA":"L","CTC":"L","CTG":"L","CTT":"L",
    "CCA":"P","CCC":"P","CCG":"P","CCT":"P",
    "CAC":"H","CAT":"H","CAA":"Q","CAG":"Q",
    "CGA":"R","CGC":"R","CGG":"R","CGT":"R",
    "GTA":"V","GTC":"V","GTG":"V","GTT":"V",
    "GCA":"A","GCC":"A","GCG":"A","GCT":"A",
    "GAC":"D","GAT":"D","GAA":"E","GAG":"E",
    "GGA":"G","GGC":"G","GGG":"G","GGT":"G",
    "TCA":"S","TCC":"S","TCG":"S","TCT":"S",
    "TTC":"F","TTT":"F","TTA":"L","TTG":"L",
    "TAC":"Y","TAT":"Y","TAA":"*","TAG":"*","TGA":"*"

}

STANDARD_AAs = set("ACDEFGHIKLMNPQRSTVWY")

WINDOW_SIZE = 9  


def translate(dna):

    protein = ""

    for i in range(0, len(dna) - 2, 3):

        codon = dna[i:i + 3]
        amino_acid = genetic_code.get(codon, "X")

        if amino_acid == "*":
            break

        protein += amino_acid

    return protein



#  CDS Retrieval

In [18]:
def get_cds(gene, cds_cache):

    if gene in cds_cache:
        return cds_cache[gene]

    print(f"Retrieving CDS for {gene}")

    time.sleep(0.34)   

    try:
        search = Entrez.esearch(
            db="nucleotide",
            term=f"{gene}[Gene] AND Homo sapiens[Organism] AND biomol_mrna[PROP] AND srcdb_refseq[PROP]"
        )
        result = Entrez.read(search)

        if len(result["IdList"]) == 0:
            print("No RefSeq mRNA found for", gene)
            cds_cache[gene] = None
            return None

        for nucleotide_id in result["IdList"]:

            time.sleep(0.34)

            handle = Entrez.efetch(
                db="nucleotide", id=nucleotide_id, rettype="gb", retmode="text"
            )
            record = SeqIO.read(handle, "genbank")

            cds = None
            for feature in record.features:
                if feature.type == "CDS":
                    cds = feature.extract(record.seq)
                    break

            if cds is not None:
                print("Using:", record.id, "for", gene)
                cds_str = str(cds).upper()
                cds_cache[gene] = cds_str
                return cds_str

        print("No CDS found for", gene)
        cds_cache[gene] = None
        return None

    except Exception as error:
        print("NCBI error for", gene, ":", error)
        cds_cache[gene] = None
        return None


# Peptide property Calculation

In [19]:

def calculate_peptide_properties(peptide):

    if not set(peptide).issubset(STANDARD_AAs):
        return None

    analysis = ProteinAnalysis(peptide)

    mw = analysis.molecular_weight()
    gravy = analysis.gravy()
    aromaticity = analysis.aromaticity()
    instability = analysis.instability_index()
    pI = analysis.isoelectric_point()

    if hasattr(analysis, "get_amino_acids_percent"):
        aa_percent = analysis.get_amino_acids_percent()
    else:
        aa_percent = analysis.amino_acids_percent

    aa_percent_total = sum(aa_percent.values())
    if aa_percent_total > 0:
        aa_percent = {aa: value / aa_percent_total for aa, value in aa_percent.items()}

    hydrophobic = aa_percent["A"] + aa_percent["V"] + aa_percent["I"] + \
                   aa_percent["L"] + aa_percent["M"] + aa_percent["F"] + \
                   aa_percent["W"] + aa_percent["Y"]

    polar = aa_percent["S"] + aa_percent["T"] + aa_percent["N"] + \
            aa_percent["Q"] + aa_percent["C"]

    positive = peptide.count("K") + peptide.count("R") + peptide.count("H")
    negative = peptide.count("D") + peptide.count("E")
    net_charge = positive - negative

    return {

        "Molecular_weight": round(mw, 2),
        "GRAVY": round(gravy, 3),
        "Aromaticity": round(aromaticity, 3),
        "Instability_index": round(instability, 2),
        "Isoelectric_point": round(pI, 2),
        "Hydrophobic_fraction": round(hydrophobic, 3),
        "Polar_fraction": round(polar, 3),
        "Net_charge": net_charge,

    }


def search_human_proteome(peptide, human_proteins):

    for protein_id, sequence in human_proteins:
        if peptide in sequence:
            return protein_id

    return None



# Process fusions

In [20]:
def process_fusion(dna_fs, gene_a_cds, gene_b_cds, reported_frame,
                    gene_a_name, gene_b_name, sample_id, tool, human_proteins):

    gene_a = gene_a_cds.replace(" ", "").upper()
    gene_b = gene_b_cds.replace(" ", "").upper()
    dna_fs = dna_fs.replace(" ", "").upper()

    if "*" not in dna_fs:
        return [], "no breakpoint marker in dna_fs"

    left_fragment, right_fragment = dna_fs.split("*")

    left_in_a = gene_a.find(left_fragment)
    left_in_b = gene_b.find(left_fragment)
    right_in_a = gene_a.find(right_fragment)
    right_in_b = gene_b.find(right_fragment)

    if left_in_a != -1 and right_in_b != -1:
        left_sequence = gene_a[:left_in_a + len(left_fragment)]
        right_sequence = gene_b[right_in_b:]
        upstream_gene_name, downstream_gene_name = "Gene A", "Gene B"
        downstream_index = right_in_b

    elif left_in_b != -1 and right_in_a != -1:
        left_sequence = gene_b[:left_in_b + len(left_fragment)]
        right_sequence = gene_a[right_in_a:]
        upstream_gene_name, downstream_gene_name = "Gene B", "Gene A"
        downstream_index = right_in_a

    else:
        return [], "unable to reconstruct fusion CDS (fragments not found)"

    fusion_cds = left_sequence + right_sequence
    junction = len(left_sequence)

    protein = translate(fusion_cds)
    gene_a_protein = translate(gene_a_cds)
    gene_b_protein = translate(gene_b_cds)

    junction_aa = junction // 3

    if junction_aa >= len(protein):
        return [], "junction falls beyond translated protein (early stop codon)"

    # Frame status: two independent methods 

    calculated_frame_math = "in-frame" if junction % 3 == downstream_index % 3 else "out-of-frame"

    downstream_native_protein = gene_b_protein if downstream_gene_name == "Gene B" else gene_a_protein
    fusion_downstream_protein = protein[junction_aa:]
    downstream_junction_aa = downstream_index // 3
    native_downstream_protein = downstream_native_protein[downstream_junction_aa:]

    comparison_length = min(30, len(fusion_downstream_protein), len(native_downstream_protein))

    if comparison_length > 0:
        matches = sum(
            1 for i in range(comparison_length)
            if fusion_downstream_protein[i] == native_downstream_protein[i]
        )
        identity = matches / comparison_length
    else:
        identity = 0.0

    calculated_frame_identity = "in-frame" if identity >= 0.90 else "out-of-frame"
    frame_methods_agree = (calculated_frame_math == calculated_frame_identity)
    calculated_frame = calculated_frame_math

    #  Candidate peptide windows (novel region only)  

    if calculated_frame == "in-frame":
        novel_start_aa, novel_end_aa = junction_aa, junction_aa
    else:
        novel_start_aa, novel_end_aa = junction_aa, len(protein) - 1

    earliest_start = max(0, novel_start_aa - WINDOW_SIZE + 1)
    latest_start = min(len(protein) - WINDOW_SIZE, novel_end_aa)

    candidate_peptides = []
    for start in range(earliest_start, latest_start + 1):
        peptide = protein[start:start + WINDOW_SIZE]
        candidate_peptides.append((start, peptide))

    junction_peptides, downstream_peptides = [], []

    for start, peptide in candidate_peptides:
        end = start + WINDOW_SIZE - 1
        if start <= junction_aa <= end:
            junction_peptides.append((start, peptide))
        elif calculated_frame == "out-of-frame" and start > junction_aa:
            downstream_peptides.append((start, peptide))

    # Empirical novelty confirmation (absent from both native proteins) 

    novel_peptides = []

    for start, peptide in junction_peptides:
        if peptide not in gene_a_protein and peptide not in gene_b_protein:
            novel_peptides.append((start, peptide, "junction"))

    for start, peptide in downstream_peptides:
        if peptide not in gene_a_protein and peptide not in gene_b_protein:
            novel_peptides.append((start, peptide, "downstream"))

    if len(novel_peptides) == 0:
        return [], "no novel peptides survived (all matched native gene A/B protein)"

    # Properties + proteome check + annotation 

    peptide_records = []

    for start, peptide, kind in novel_peptides:

        peptide_properties = calculate_peptide_properties(peptide)

        protein_match = search_human_proteome(peptide, human_proteins)
        human_novel = "YES" if protein_match is None else "NO"

        record = {

            "Sample_ID": sample_id,
            "Tool": tool,
            "GeneA_name": gene_a_name,
            "GeneB_name": gene_b_name,
            "Upstream_gene": upstream_gene_name,
            "Downstream_gene": downstream_gene_name,
            "Reported_frame": reported_frame,
            "Calculated_frame_math": calculated_frame_math,
            "Calculated_frame_identity": calculated_frame_identity,
            "Frame_methods_agree": frame_methods_agree,
            "Peptide_type": kind,
            "Peptide": peptide,
            "Peptide_length": len(peptide),
            "AA_position": start,
            "DNA_junction": junction,
            "AA_junction": junction_aa,
            "Fusion_sequence": dna_fs,
            "Novel_to_human": human_novel,
            "Matching_protein": protein_match,

        }

        if peptide_properties is not None:
            record.update(peptide_properties)
        else:
            record["Properties_calculated"] = False

        peptide_records.append(record)

    return peptide_records, None



# Main pipeline

In [21]:
if __name__ == "__main__":

    aggregate_db_path = input("Enter path to all_fusions_database.tsv: ")
    sample_folder_path = input("Enter path to the folder of per-sample fusion TSVs: ")
    proteome_fasta_path = input("Enter path to human_sprot.fasta: ")

    # Step 1: filter the aggregate database (frequency >= 3, both frame types) 

    agg_df = pd.read_csv(aggregate_db_path, sep="\t")
    agg_df = agg_df[agg_df["frequency"] >= 3]

    print("Qualifying fusions in aggregate database:", len(agg_df))

    qualifying_pairs = set()

    for _, row in agg_df.iterrows():
        parts = str(row["canonical_pair"]).split("|")
        if len(parts) == 2:
            gene_x, gene_y = parts[0].strip(), parts[1].strip()
            qualifying_pairs.add(frozenset([gene_x, gene_y]))

    print("Unique qualifying gene pairs:", len(qualifying_pairs))

    # Step 2: load and clean the per-sample files 

    master_fusion_table = load_all_sample_fusions(sample_folder_path)
    print("Total per-sample fusion records loaded:", len(master_fusion_table))

    
    # Step 3: join to keep only rows whose gene pair qualified, and that parsed successfully 

    def pair_qualifies(row):
        return frozenset([row["Gene_A_name"], row["Gene_B_name"]]) in qualifying_pairs

    mask = master_fusion_table.apply(pair_qualifies, axis=1) & master_fusion_table["Parse_successful"]
    filtered_table = master_fusion_table[mask].reset_index(drop=True)

    print("Rows after joining with aggregate database filter:", len(filtered_table))

    # Step 4: load the human proteome once 

    print("Loading Swiss-Prot human proteome...")
    human_proteins = [
        (record.id, str(record.seq))
        for record in SeqIO.parse(proteome_fasta_path, "fasta")
    ]
    print("Proteins loaded:", len(human_proteins))

    # Step 5: process every qualifying fusion, fetching CDS with caching

    cds_cache = {}
    all_results = []
    failures = []

    for i, row in filtered_table.iterrows():

        print(f"\n[{i + 1}/{len(filtered_table)}] {row['Sample_ID']} | {row['Tool']} | "
              f"{row['Gene_A_name']}-{row['Gene_B_name']}")

        gene_a_cds = get_cds(row["Gene_A_name"], cds_cache)
        gene_b_cds = get_cds(row["Gene_B_name"], cds_cache)

        if gene_a_cds is None or gene_b_cds is None:
            failures.append({
                "Sample_ID": row["Sample_ID"], "Tool": row["Tool"],
                "Gene_A_name": row["Gene_A_name"], "Gene_B_name": row["Gene_B_name"],
                "Reason": "CDS retrieval failed for one or both genes",
            })
            continue

        peptide_records, failure_reason = process_fusion(
            dna_fs=row["dna_fs"],
            gene_a_cds=gene_a_cds,
            gene_b_cds=gene_b_cds,
            reported_frame=row["reported_frame"],
            gene_a_name=row["Gene_A_name"],
            gene_b_name=row["Gene_B_name"],
            sample_id=row["Sample_ID"],
            tool=row["Tool"],
            human_proteins=human_proteins,
        )

        if failure_reason is not None:
            failures.append({
                "Sample_ID": row["Sample_ID"], "Tool": row["Tool"],
                "Gene_A_name": row["Gene_A_name"], "Gene_B_name": row["Gene_B_name"],
                "Reason": failure_reason,
            })
            continue

        all_results.extend(peptide_records)

    # Step 6: export

    results_df = pd.DataFrame(all_results)
    results_df.to_csv("Final_candidate_peptides.csv", index=False)

    failures_df = pd.DataFrame(failures)
    failures_df.to_csv("Failed_fusions_log.csv", index=False)

    print("PIPELINE COMPLETE")
    print("Fusions processed successfully:", len(filtered_table) - len(failures))
    print("Fusions failed/skipped:", len(failures))
    print("Total candidate peptides:", len(results_df))
    print("Saved: Final_candidate_peptides.csv")
    print("Saved: Failed_fusions_log.csv")

Enter path to all_fusions_database.tsv:  /home/jerrybryt/Downloads/all_fusions_database(1).tsv
Enter path to the folder of per-sample fusion TSVs:  /home/jerrybryt/MSBT/IP/Gene_Fusions/Case_specific_fusions/Case_specific_fusions
Enter path to human_sprot.fasta:  /home/jerrybryt/MSBT/IP/Gene_Fusions/Somatic-Gene-Fusions/human_sprot.fasta


Qualifying fusions in aggregate database: 52
Unique qualifying gene pairs: 52
Total per-sample fusion records loaded: 1975
Rows after joining with aggregate database filter: 360
Loading Swiss-Prot human proteome...
Proteins loaded: 20431

[1/360] AKU040 | fusioncatcher | POM121C-RBAK
Retrieving CDS for POM121C
Using: NM_001099415.3 for POM121C
Retrieving CDS for RBAK
Using: NM_001204456.2 for RBAK

[2/360] AKU040 | fusioncatcher | POM121C-RBAK

[3/360] AKU040 | fusioncatcher | AC138649.1-HERC2
Retrieving CDS for AC138649.1
No RefSeq mRNA found for AC138649.1
Retrieving CDS for HERC2
Using: NM_004667.6 for HERC2

[4/360] AKU040 | fusioncatcher | AC138649.1-HERC2

[5/360] AKU040 | fusioncatcher | AC138649.1-HERC2

[6/360] AKU040 | fusioncatcher | AC138649.1-HERC2

[7/360] AKU040 | fusioncatcher | FOXP1-ETV6
Retrieving CDS for FOXP1
Using: NM_001349340.3 for FOXP1
Retrieving CDS for ETV6
Using: NM_001413915.1 for ETV6

[8/360] AKU040 | fusioncatcher | HERC2-AC138649.1

[9/360] AKU040 | fu

/home/jerrybryt/anaconda3/lib/python3.11/site-packages/Bio/SeqUtils/ProtParam.py:106: BiopythonDeprecationWarning: The get_amino_acids_percent method has been deprecated and will likely be removed from Biopython in the near future. Please use the amino_acids_percent attribute instead.
  warnings.warn(


Using: NM_001015509.3 for PTRH2
Retrieving CDS for IKZF3
Using: XM_054315483.1 for IKZF3

[167/360] KAIC009 | fusioncatcher | PTRH2-IKZF3

[168/360] KAIC009 | fusioncatcher | PTRH2-IKZF3

[169/360] KAIC009 | fusioncatcher | BGN-LNPEP
Retrieving CDS for BGN
Using: NM_001711.6 for BGN
Retrieving CDS for LNPEP
Using: XM_054352582.1 for LNPEP

[170/360] KAIC009 | fusioncatcher | BGN-LNPEP

[171/360] KAIC009 | fusioncatcher | BGN-LNPEP

[172/360] KAIC009 | fusioncatcher | BGN-LNPEP

[173/360] KAIC009 | fusioncatcher | LNPEP-BGN

[174/360] KAIC009 | fusioncatcher | RUNX2-ATXN2

[175/360] KAIC009 | fusioncatcher | PRRX1-BCL9

[176/360] KAIC009 | fusioncatcher | U73166.1-AL157938.2

[177/360] KAIC009 | fusioncatcher | FGFR2-ETV6

[178/360] KAIC009 | fusioncatcher | FOXP1-ETV6

[179/360] KAIC009 | fusioncatcher | NUMBL-MAML2

[180/360] KAIC009 | fusioncatcher | ST6GALNAC5-THAP11

[181/360] KAIC009 | fusioncatcher | TAF15-TCF3

[182/360] KAIC009 | fusioncatcher | TCF3-TAF15

[183/360] KAIC010 | 

In [22]:
overview = pd.read_csv("Final_candidate_peptides.csv")

In [23]:
print(overview.head())

  Sample_ID           Tool GeneA_name GeneB_name Upstream_gene  \
0    AKU089  fusioncatcher      AGBL2   ANKRD30A        Gene A   
1    AKU089  fusioncatcher      AGBL2   ANKRD30A        Gene A   
2    AKU089  fusioncatcher      AGBL2   ANKRD30A        Gene A   
3    AKU089  fusioncatcher      AGBL2   ANKRD30A        Gene A   
4    AKU089  fusioncatcher      AGBL2   ANKRD30A        Gene A   

  Downstream_gene Reported_frame Calculated_frame_math  \
0          Gene B       in-frame              in-frame   
1          Gene B       in-frame              in-frame   
2          Gene B       in-frame              in-frame   
3          Gene B       in-frame              in-frame   
4          Gene B       in-frame              in-frame   

  Calculated_frame_identity  Frame_methods_agree  ... Matching_protein  \
0                  in-frame                 True  ...              NaN   
1                  in-frame                 True  ...              NaN   
2                  in-frame     

In [24]:
print(overview)

    Sample_ID           Tool GeneA_name GeneB_name Upstream_gene  \
0      AKU089  fusioncatcher      AGBL2   ANKRD30A        Gene A   
1      AKU089  fusioncatcher      AGBL2   ANKRD30A        Gene A   
2      AKU089  fusioncatcher      AGBL2   ANKRD30A        Gene A   
3      AKU089  fusioncatcher      AGBL2   ANKRD30A        Gene A   
4      AKU089  fusioncatcher      AGBL2   ANKRD30A        Gene A   
..        ...            ...        ...        ...           ...   
215   KAIC029  fusioncatcher      EIF3H      TRPS1        Gene A   
216   KAIC029  fusioncatcher      EIF3H      TRPS1        Gene A   
217   KAIC029  fusioncatcher      EIF3H      TRPS1        Gene A   
218   KAIC029  fusioncatcher      EIF3H      TRPS1        Gene A   
219   KAIC029  fusioncatcher      EIF3H      TRPS1        Gene A   

    Downstream_gene Reported_frame Calculated_frame_math  \
0            Gene B       in-frame              in-frame   
1            Gene B       in-frame              in-frame   
2  